Вы работаете аналитиком данных в онлайн-кинотеатре *SkyCinema*.\
Сегодня вы будете работать с **ивентами** (event - *событие*). Ивенты - это записи о клиентских событиях, таких как включение плеера с видео или заход на сайт онлайн-кинотеатра. Одной из важнейших задач аналитики является изучение и правка ошибочных ивентов.

Ваша задача - изучить датасет на наличие ошибок.

Импортируйте файл *event_list.xlsx* и сохраните его в датафрейм.

**Описание данных**

    - account_id - идентификатор пользователя
    - server_time - время совершения действия
    - action_id - идентификатор (тег) действия
    - device_type - тип устройства
    - user_browser - браузер
    - screen_type - тип страницы, на которой совершается действие.
*В рамках данного технического задания вам необходимо взять только те строки, у которых колонка screen_type принимает значение "player".*
    
**Описание ивентов**

    - 'screenview' - заход на страницу с проигрывателем тайтла
    - 'click' - клик по плееру
    - 'playback_start' - включение проигрывателя
    - 'playback_pause' - постановка проигрывателя на паузу
    - 'playback_unpause' - снятия проигрывателя с паузы
    - 'playback_stop'  - выключение страницы с проигрывателем

In [5]:
import pandas as pd
from IPython.display import display
df = pd.read_excel('event_list.xlsx')
df = df[df['screen_type'] == 'player']




### Задача 1

Выделите 10 самых активных пользователей (тех, что сделали больше всего действий за данный период времени).\
Каким браузером чаще всего пользовались эти 10 пользователей?\
Какое действие занимает наибольший процент среди всех действий этих 10 пользователей?

In [6]:
us_act = df.groupby('account_id')['screen_type'].count().reset_index(name='cnt_scr')
top10 = us_act.sort_values('cnt_scr', ascending=False).head(10)
print('Топ-10 самых активных пользователей:')
display(top10)

browser_us = df[df['account_id'].isin(top10['account_id'])].groupby('user_browser')['screen_type'].count().reset_index(name='cnt_br')
top1_br = browser_us.sort_values('cnt_br', ascending=False).head(1)
print('Браузер, который чаще всего использовался топ-10 пользователями:')
display(top1_br)
act_cnt = df[df['account_id'].isin(top10['account_id'])].groupby('action_id')['account_id'].count().reset_index(name='cnt_act')
total_act = act_cnt['cnt_act'].sum()
act_cnt['percent'] = (act_cnt['cnt_act'] / total_act) * 100
display(act_cnt.sort_values('percent', ascending=False).head(1))

Топ-10 самых активных пользователей:


,account_id,cnt_scr
36,102598,1004
145,103398,386
37,102605,288
76,102904,142
108,103149,141
199,103804,128
120,103219,128
10,102405,116
74,102883,107
138,103364,102


Браузер, который чаще всего использовался топ-10 пользователями:


,user_browser,cnt_br
4,Yandex,1004


,action_id,cnt_act,percent
0,click,1595,62.745869


### Задача 2

Проверьте данные на состоятельность:

1. Есть ли клиенты, которые снимали видео с паузы хотя бы раз, но при этом не ставили видео на паузу ни разу?
2. Есть ли клиенты, которые ставили видео на паузу хотя бы раз, но при этом не включали проигрыватель?

С каких устройств и браузеров заходят пользователи, у которых встречаются подобные аномалии?

*Подсказка*\
Воспользуйтесь методом серии unique(), чтобы для каждого действия (ивента) создать списки с уникальными пользователями, которые совершали эти действия. \
Посмотрите на соответствующие исключения списков друг из друга, чтобы определить пользователей с проблемной последовательностью событий.

In [7]:

users_res = df[df['action_id'] == 'playback_unpause']['account_id'].unique()
users_paus = df[df['action_id'] == 'playback_pause']['account_id'].unique()
users_pl = df[df['action_id'] == 'playback_start']['account_id'].unique()
users_resumed_no_paused = set(users_res) - set(users_paus)
users_paused_no_played = set(users_paus) - set(users_pl)
anomalous_users_resumed_no_paused = df[df['account_id'].isin(users_resumed_no_paused)][['account_id', 'user_browser', 'device_type']]
anomalous_users_paused_no_played = df[df['account_id'].isin(users_paused_no_played)][['account_id', 'user_browser', 'device_type']]
print('Пользователи, которые снимали видео с паузы хотя бы раз, но не ставили его на паузу:')
display(anomalous_users_resumed_no_paused.drop_duplicates())

print('\nПользователи, которые ставили видео на паузу хотя бы раз, но не включали проигрыватель:')
display(anomalous_users_paused_no_played.drop_duplicates())

Пользователи, которые снимали видео с паузы хотя бы раз, но не ставили его на паузу:


,account_id,user_browser,device_type
283,102345,Safari,desktop_web
4733,102918,Chrome,desktop_web
5264,103050,Chrome,desktop_web



Пользователи, которые ставили видео на паузу хотя бы раз, но не включали проигрыватель:


,account_id,user_browser,device_type
3737,102669,Safari,desktop_web
5294,103052,Safari,desktop_web


### Задача 3

1. Изучите пользователей, у которых есть только одно событие. Какой вид события встречается чаще всего? Какие из встречающихся событий вы бы посчитали ошибочными?
2. Изучите пользователей, у которых есть только два события. Какие из их последовательностей событий вы бы посчитали ошибочными?

In [8]:
fr_event_us = df.groupby('account_id').filter(lambda x: len(x) == 1)
fr_event_cnt = fr_event_us['action_id'].value_counts()
ev_name_max = fr_event_cnt.idxmax()
ev_num_max = fr_event_cnt.max()
print('Пользователи с одним событием:')
display(fr_event_us)
print(f'Наиболее частое событие: {ev_name_max} Количество: {ev_num_max}')
er_fr_event = fr_event_us[fr_event_us['action_id'].isin(['playback_stop', 'playback_pause', 'playback_unpause'])]
print('Пользователи с одним событием и ошибочным действием:')
display(er_fr_event)


Пользователи с одним событием:


,account_id,server_time,screen_type,action_id,device_type,user_browser
900,102461,2021-10-01 07:22:42,player,click,desktop_web,Safari
1444,102520,2021-10-01 19:51:25,player,playback_stop,desktop_web,Chrome
4395,102839,2021-10-01 15:32:53,player,playback_start,desktop_web,Opera
4439,102853,2021-10-01 08:55:10,player,click,desktop_web,Yandex
4440,102865,2021-10-01 05:55:31,player,playback_stop,desktop_web,Safari
5843,103140,2021-10-01 21:02:42,player,playback_start,mobile_web,Mobile Safari
9920,103813,2021-10-01 07:17:19,player,playback_stop,desktop_web,Chrome


Наиболее частое событие: playback_stop Количество: 3
Пользователи с одним событием и ошибочным действием:


,account_id,server_time,screen_type,action_id,device_type,user_browser
1444,102520,2021-10-01 19:51:25,player,playback_stop,desktop_web,Chrome
4440,102865,2021-10-01 05:55:31,player,playback_stop,desktop_web,Safari
9920,103813,2021-10-01 07:17:19,player,playback_stop,desktop_web,Chrome


In [17]:
sec_event= df.groupby('account_id').filter(lambda x: len(x)==2)
invalid_sequences = []
for account_id, group in sec_event.groupby('account_id'):
    actions = group['action_id'].tolist()
    times = group['server_time'].tolist()
    if actions[0] == actions[1]:
        invalid_sequences.append((account_id, actions, times, "Дубликаты действий"))
    if actions[0] == 'playback_stop' and actions[1] != 'playback_start':
        invalid_sequences.append((account_id, actions, times, "Не логичные действия"))
    if actions[0] == 'click' and len(actions) > 1 and actions[1] not in ['playback_start', 'playback_stop']:
        invalid_sequences.append((account_id, actions, times, "Подозрительные действия"))
invalid_df = pd.DataFrame(invalid_sequences, columns=['account_id', 'actions', 'times', 'issue'])
merged_df = sec_event.merge(invalid_df, on='account_id', how='inner')
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]
merged_df = merged_df.drop_duplicates(subset=['account_id', 'issue'])
print("Найденные ошибочные последовательности:")
display(merged_df)
                                        

Найденные ошибочные последовательности:


,account_id,server_time,screen_type,action_id,device_type,user_browser,actions,times,issue
0,102682,2021-10-01 00:06:08,player,playback_stop,desktop_web,Chrome,"[playback_stop, playback_stop]","[2021-10-01 00:06:08, 2021-10-01 00:10:12]",Дубликаты действий
1,102682,2021-10-01 00:06:08,player,playback_stop,desktop_web,Chrome,"[playback_stop, playback_stop]","[2021-10-01 00:06:08, 2021-10-01 00:10:12]",Не логичные действия
4,103052,2021-10-01 00:06:40,player,click,desktop_web,Safari,"[click, playback_pause]","[2021-10-01 00:06:40, 2021-10-01 00:06:53]",Подозрительные действия
